# 03 · The ini file and the grid types

A setup starts from one ini file, read into one checked `GridConfig` (`posydon/grids/setup/config.py`). This notebook covers:

- the **grid-type registry** (`grid_types.py`): the six grid types and what makes each different;
- the **starter ini** (`template.py`, `posydon-grid init`);
- the **new ini format**, section by section, and how it maps onto `GridConfig`;
- the **errors**: each one names the section and key;
- **new vs legacy**: how the setup tells them apart (`steps.is_new_format`, `read_any_config`).

In [1]:
from nb_env import env, fresh, write, show, grid_csv
ENV = env()

import pandas as pd
from posydon.grids.setup.grid_types import GRID_TYPES, describe_grid_types, get_grid_type
from posydon.grids.setup.template import ini_template
from posydon.grids.setup.config import ConfigError, read_config

## The grid types

Every grid type is one `GridType` entry. Its fields say how each star starts: from a ZAMS model (`zams`), built in formation steps (`formation`), a compact object MESA does not evolve (`point_mass`), or no second star (`None`).

In [2]:
pd.DataFrame([vars(t) for t in GRID_TYPES.values()]).set_index("name")

,description,folder,star1,star2,single_star,single_star_flag
name,,,,,,
HMS-HMS,two hydrogen-rich main-sequence stars,None,zams,zams,False,False
CO-HMS_RLO,a compact object and a hydrogen-rich main-sequ...,CO-H_star,zams,point_mass,False,False
CO-HeMS,a compact object and a helium main-sequence star,CO-He_star,formation,point_mass,False,False
HeMS-HMS,a helium and a hydrogen-rich main-sequence star,HeMS-HMS,formation,zams,False,False
single_HMS,single hydrogen-rich main-sequence stars,None,zams,None,True,True
single_HeMS,single helium main-sequence stars,CO-He_star,formation,None,True,True


`folder` is only used for legacy scenarios (the old layout of the inlist repository). New releases list each grid type's files in a manifest instead (notebook 04).

Unknown names are refused with the list of known ones. `MS-MS`, used by the main v2 HMS-HMS ini files, is not a grid type; legacy inis that say it are read as HMS-HMS with a note (tracker G4).

In [3]:
try:
    get_grid_type("MS-MS")
except ConfigError as error:
    print("ConfigError:", error)

ConfigError: [grid] type: unknown grid type 'MS-MS'; expected one of HMS-HMS, CO-HMS_RLO, CO-HeMS, HeMS-HMS, single_HMS, single_HeMS


## The starter ini

`ini_template` is what `posydon-grid init` writes. It differs per grid type in the ZAMS line and in what is written for each star:

In [4]:
import difflib
a, b = ini_template("HMS-HMS").splitlines(), ini_template("CO-HeMS").splitlines()
print("\n".join(difflib.unified_diff(a, b, "HMS-HMS", "CO-HeMS", lineterm="", n=0)))

--- HMS-HMS
+++ CO-HeMS
@@ -1,3 +1,3 @@
-; posydon-grid settings for a HMS-HMS grid:
-; two hydrogen-rich main-sequence stars.
-; Written by posydon-grid init HMS-HMS. Paths are relative to this file.
+; posydon-grid settings for a CO-HeMS grid:
+; a compact object and a helium main-sequence star.
+; Written by posydon-grid init CO-HeMS. Paths are relative to this file.
@@ -8 +8 @@
-type = HMS-HMS
+type = CO-HeMS
@@ -12 +11,0 @@
-zams = <ZAMS model, e.g. POSYDON-MESA-INLISTS/r11701/ZAMS_models/zams_z1.42m2_y0.2703.data>
@@ -29 +28 @@
-star2 = history, final_model
+star2 = none


In [ ]:
print(ini_template("single_HeMS"))

## The new format, section by section

| Section | Goes to | What it says |
|---|---|---|
| `[grid]` | `config.grid` (`GridSection`) | grid type, fixed or dynamic, the points, the run directory, ZAMS models |
| `[scenario]` | `config.scenario` (`ScenarioSection`) | which POSYDON inlists: `release` and `rerun`, read from the clone `inlist_repo` as it is |
| `[mesa_inlists]`, `[posydon_inlists]`, `[user_inlists]` | `config.inlists[layer]` (`InlistFiles`) | files per target, for each layer |
| `[output]` | `config.output` | history, final profile, final model per star; `history_interval` |
| `[submission]` | `config.submission` | `shell`, `slurm_array` or `slurm_mpi`, and the SLURM settings |
| `[extras]` | `config.extras` | Fortran sources and makefiles, by their legacy `[mesa_extras]` names |

Usually you only write `[grid]`, `[scenario]`, maybe `[user_inlists]`, `[output]` and `[submission]`. MESA's defaults and the stock Fortran sources are filled in from the release (notebook 05).

In [ ]:
folder = fresh("03_ini")
grid_csv(folder)
write(folder / "my_inlist1", "&controls\n    mesh_delta_coeff = 0.8d0\n/\n")
write(folder / "my_step0", "&controls\n    max_age = 1d6\n/\n")
ini = write(folder / "grid.ini", f"""
    [grid]
    type = HeMS-HMS
    points = grid.csv                 ; relative to this file
    zams_star2 = {ENV.inlists}/r11701/ZAMS_models/zams_z1.42m2_y0.2703.data

    [scenario]
    release = v2.0
    rerun = matthias_lbv_thermo_dedt_noAgeLimit
    inlist_repo = {ENV.inlists}

    [user_inlists]
    star1 = my_inlist1                ; shorthand for star1_controls and star1_job
    star1_formation_step0 = my_step0  ; on top of formation step 0 only

    [output]
    star1 = history
    star2 = history, final_profile

    [submission]
    backend = slurm_array
    account = my_account
    partition = public-cpu
    walltime = 1-00:00:00
    """)
config = read_config(ini)
config.grid

In [7]:
config.scenario

ScenarioSection(source='path', release='v2.0', rerun='matthias_lbv_thermo_dedt_noAgeLimit', mesa=None, ref=None, commit=None, branch=None, system_type=None, inlist_repo=PosixPath('/Users/max/Documents/POSYDON-MESA-INLISTS-v2-base'))

In [8]:
config.inlists["user_inlists"]

InlistFiles(targets={'star1_controls': [PosixPath('/Users/max/Documents/POSYDON/.claude/worktrees/posydon-grid-setup-refactor-7d725a/grid_rewrite_tests/notebooks/_work/03_ini/my_inlist1')], 'star1_job': [PosixPath('/Users/max/Documents/POSYDON/.claude/worktrees/posydon-grid-setup-refactor-7d725a/grid_rewrite_tests/notebooks/_work/03_ini/my_inlist1')], 'star1_formation_step0_controls': [PosixPath('/Users/max/Documents/POSYDON/.claude/worktrees/posydon-grid-setup-refactor-7d725a/grid_rewrite_tests/notebooks/_work/03_ini/my_step0')], 'star1_formation_step0_job': [PosixPath('/Users/max/Documents/POSYDON/.claude/worktrees/posydon-grid-setup-refactor-7d725a/grid_rewrite_tests/notebooks/_work/03_ini/my_step0')]}, whole_files={})

Note how the shorthand `star1 = ...` became two targets, and `star1_formation_step0` became the controls and job targets of step 0. The other layers are empty: the release fills them in later.

In [9]:
config.output, config.submission.backend

(OutputSection(history_interval=1, binary_history=True, star1=('history',), star2=('history', 'final_profile')),
 'slurm_array')

`${section:key}` interpolation works, and `;` starts a comment:

In [ ]:
ini2 = write(folder / "interp.ini", f"""
    [grid]
    type = HMS-HMS
    points = grid.csv
    zams = ${{scenario:inlist_repo}}/r11701/ZAMS_models/zams_z1.42m2_y0.2703.data

    [scenario]
    release = v2.0
    inlist_repo = {ENV.inlists}
    """)
print(read_config(ini2).grid.zams_star1)

## Errors name the section and the key

Every problem is a `ConfigError` (a `ValueError`) that says where it is. A few on purpose:

In [ ]:
cases = {
    "a typo in a key": "[grid]\ntype = HMS-HMS\npoints = grid.csv\nrun_dir = run\n",
    "an unknown grid type": "[grid]\ntype = MS-MS\npoints = grid.csv\n",
    "dynamic without psy-cris": "[grid]\ntype = HMS-HMS\nmode = dynamic\npoints = grid.csv\n",
    "a ref, from before": "[grid]\ntype = HMS-HMS\npoints = grid.csv\n[scenario]\nrelease = v2.0\nref = v2_sources\n",
    "a rerun without a release": "[grid]\ntype = HMS-HMS\npoints = grid.csv\n[scenario]\nrerun = x\n",
    "a clone without a release": "[grid]\ntype = HMS-HMS\npoints = grid.csv\n[scenario]\ninlist_repo = ~/inlists\n",
    "a bad backend": "[grid]\ntype = HMS-HMS\npoints = grid.csv\n[submission]\nbackend = pbs\n",
    "an unknown section": "[grid]\ntype = HMS-HMS\npoints = grid.csv\n[slurm]\naccount = x\n",
    "two column lists": "[grid]\ntype = HMS-HMS\npoints = grid.csv\n[user_inlists]\nhistory_columns = a.list, b.list\n",
}
for what, text in cases.items():
    path = write(folder / "bad.ini", text)
    try:
        read_config(path)
        print(f"{what:34} -> accepted")
    except ConfigError as error:
        print(f"{what:34} -> {error}")

`ref`, `commit`, `branch` and `source` are refused with a note: the setup reads the inlist repository as it is, and the release names the physics. Which version your clone holds is up to you (`git pull`); `setup_record.ini` records its commit and a fingerprint of the files used (notebook 07).

## New format or legacy?

An ini with a `[grid]` section is new-format. Anything else is a legacy ini (today's `[slurm]`, `[mesa_inlists]`, `[mesa_extras]`, `[run_parameters]`), read by `legacy.py` into the same `GridConfig` (notebook 08).

A new-format ini says everything itself, so `posydon-grid setup` refuses the legacy command-line options for it:

In [12]:
from posydon.grids.setup.steps import is_new_format, read_any_config
print(is_new_format(ini))
try:
    read_any_config(ini, grid_type="fixed")
except ValueError as error:
    print("ValueError:", error)

True
ValueError: --grid-type only applies to legacy ini files; a new-format ini says this in [grid] mode, run_directory and [submission] backend


The deprecated `posydon-setup-grid` still accepts them (`overrides=True`), and then they win over the ini:

In [13]:
c = read_any_config(ini, submission_type="shell", overrides=True)
c.submission.backend

'shell'

## Try it

- Write an ini for `single_HMS` and look at `config.inlists`: what happens to `star1 = ...` for a single-star grid?
- Add `[posydon_inlists] star1 = my_physics_inlist` and see where it goes. (It is applied *before* the scenario's files within `posydon_inlists`; notebook 05 shows the order.)